In [47]:
import cv2
import numpy as np
import glob
import matplotlib.pyplot as plt
import pandas as pd
from skimage.measure import regionprops, label
from natsort import natsorted
from cellpose import plot
import trackpy as tp
import sys
sys.path.append('../defect_functions') 
from defect_pairs import * 
from average_flows import * 

# Set up matplotlib to use the Qt backend
%matplotlib qt

def calculate_neighbors(mask_image, area_threshold=60, boundary_margin=5):
    """
    Calculate neighbors and centroids for each region in the mask image.
    
    Parameters:
    - mask_image: The binary mask image where regions are labeled.
    - area_threshold: Minimum area for a region to be considered (default is 60).
    - boundary_margin: Margin from the image boundary to exclude contours (default is 5).
    
    Returns:
    - neighbors: A dictionary containing the centroids and neighboring cells for each region.
    """
    height, width = mask_image.shape
    neighbors = {}
    
    # Get region properties
    regions = regionprops(mask_image, intensity_image=mask_image)
    
    for region in regions:
        area = region.area
        color = int(region.mean_intensity)
        
        if area > area_threshold:
            mask = np.uint8(mask_image == color)
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            if contours:
                centroid = region.centroid  # Get the centroid of the region
                area = region.area
                perimeter = region.perimeter
                if color not in neighbors:
                    neighbors[color] = {'Centroid': centroid, 
                                        'Area': area, 
                                        'Perimeter': perimeter, 
                                        'Neighboring Cells': set()}

                for contour in contours:
                    # Check if the contour is within the boundary margin
                    if all(boundary_margin <= point[0][0] < width - boundary_margin and 
                           boundary_margin <= point[0][1] < height - boundary_margin for point in contour):                    
                
                    # for contour in contours:
                        for point in contour:
                            x, y = point[0]
                            # Check neighboring pixels
                            for i in range(max(0, y - 1), min(height, y + 2)):
                                for j in range(max(0, x - 1), min(width, x + 2)):
                                    if mask_image[i, j] != color and mask_image[i, j] != 0:
                                        neighbors[color]['Neighboring Cells'].add(mask_image[i, j])
    return neighbors


def create_dataframe(neighbors):
    """
    Create a DataFrame from the neighbors dictionary, excluding empty neighbors.
    
    Parameters:
    - neighbors: A dictionary containing neighboring cells and centroids.
    
    Returns:
    - A pandas DataFrame with cells, their centroids, number of neighbors, and the list of neighbors.
    """
    data = []
    for k, v in neighbors.items():
        if v['Neighboring Cells']:  # Only include if there are neighboring cells
            data.append({'Cell': k, 
                         'x': v['Centroid'][1], 
                         'y': v['Centroid'][0], 
                         'Area': v['Area'], 
                         'Perimeter': v['Perimeter'], 
                         'Neighbors Num': len(v['Neighboring Cells']), 
                         'Neighbors': list(v['Neighboring Cells'])})
    return pd.DataFrame(data)


def create_area_mask(mask_image):
    """
    Create a new mask labeled by region area.
    
    Parameters:
    - mask_image: The binary mask image where regions are labeled.
    
    Returns:
    - area_mask: A new mask where each region is labeled by its area.
    """
    # Label the regions in the mask image
    labeled_mask = label(mask_image)
    area_mask = np.zeros_like(labeled_mask, dtype=np.uint16)  # Create an empty mask for areas

    # Get region properties
    regions = regionprops(labeled_mask)

    # Assign area values to the new mask
    for region in regions:
        # area = int(255*region.eccentricity)
        area = region.area
        # Fill the area_mask with the area value for the corresponding region
        area_mask[labeled_mask == region.label] = area

    return area_mask

In [ ]:
# Main execution
im_num = 100
image_path = r"C:\Users\victo\OneDrive - BGU\DATA\Hacat\6 Wells _5x_15min\_1\Pos0\*.tif"
mask_path = r"C:\Users\victo\OneDrive - BGU\DATA\Hacat\6 Wells _5x_15min\_1\Pos0\Mask2\*.png"

img_list = natsorted(glob.glob(image_path), key=lambda y: y.lower())
masks_list = natsorted(glob.glob(mask_path), key=lambda y: y.lower())

# Load images
x, y, w, h = [0, 0, 800, 500]
raw_image = cv2.imread(img_list[im_num], cv2.IMREAD_UNCHANGED)[y:y+h, x:x+w]
mask_image = cv2.imread(masks_list[im_num], cv2.IMREAD_UNCHANGED)[y:y+h, x:x+w]    

PLOT = False
if PLOT:
    # Display overlay
    plt.figure()
    overlay = cv2.addWeighted(cv2.cvtColor(raw_image, cv2.COLOR_GRAY2RGB), 0.5, plot.mask_rgb(mask_image), 0.5, 0)
    plt.imshow(overlay)

    # Display overlay with area values
    area = [region.area for region in regionprops(mask_image)]
    perimeter = [region.perimeter for region in regionprops(mask_image)]
    area_value = [int(region.mean_intensity) for region in regionprops(mask_image, intensity_image=mask_image)]
    xy = np.array([(region.centroid[1], region.centroid[0]) for region in regionprops(mask_image)])
    plt.plot(xy[:,0], xy[:,1], 'w.', alpha=.3)
    for i in range(len(xy)):
        col = 'k' if area[i]<60 else 'w'
        plt.text(xy[i,0], xy[i,1], f"{int(area_value[i])}", color=col, fontsize=14)

# Calculate neighbors
neighbors = calculate_neighbors(mask_image)

# Create DataFrame
df_neighbors = create_dataframe(neighbors)

# # Optionally save DataFrame to CSV
# # df_neighbors.to_csv('neighbors_data.csv', index=False)

# # Display the DataFrame
# # print(df_neighbors)
# for index, row in df_neighbors.iterrows():
#     plt.text(row['Centroid'][1], row['Centroid'][0], f"{int(row['Num'])}", color='w', fontsize=12)

In [23]:
# Main execution
# Display overlay
plt.figure()
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
img_clahe = clahe.apply(raw_image)
# plt.imshow(255-img_clahe, cmap='gray')
overlay = cv2.addWeighted(cv2.cvtColor(raw_image, cv2.COLOR_GRAY2RGB), 0.5, plot.mask_rgb(mask_image), 0.3, 0)
plt.imshow(overlay)
# plt.imshow(create_area_mask(mask_image), "coolwarm")

for index, row in df_neighbors.iterrows():
    plt.text(row['x'], row['y'], f"{int(row['Neighbors Num'])}", color='w', fontsize=12)

In [3]:
img1 = cv2.imread(img_list[im_num-1], cv2.IMREAD_UNCHANGED)#[y:y+h, x:x+w]
img2 = cv2.imread(img_list[im_num], cv2.IMREAD_UNCHANGED)#[y:y+h, x:x+w]
flow = cv2.calcOpticalFlowFarneback(img1,img2, None, 0.5, 3, 
        winsize=15, iterations=3, poly_n=5, poly_sigma=1.2, flags=0) 

step = 30
x = np.arange(0, flow.shape[1], step, dtype=np.int16)
y = np.arange(0, flow.shape[0], step, dtype=np.int16)
plt.quiver(x,y, 
    flow[::step, ::step, 0], -flow[::step, ::step, 1], 
    color="k", scale=100, label="flow", alpha=.5)

In [4]:
s = 30
ori,_,_ = analyze_defects(raw_image, sigma=25)
y, x = np.mgrid[0:img1.shape[0], 0:img1.shape[1]]
quiver = plt.quiver(x[::s,::s], y[::s,::s],
    np.cos(ori)[::s,::s], np.sin(ori)[::s,::s], 
    # np.arctan2(np.sin(ori), np.cos(ori))[::s,::s],
    headaxislength=0, headwidth=0, headlength=0, width=.005, 
    scale=30, pivot='mid', alpha=.6)

In [5]:
df_neighbors.x.max(), df_neighbors.y.max()

(np.float64(1596.1768707482993), np.float64(1091.9906542056074))

In [10]:
import seaborn as sns
data = df_neighbors[df_neighbors["Area"]>60].copy()

plt.figure()
data["Shape"] = data["Perimeter"]**2/(4*np.pi*data["Area"])
sns.pairplot(data[["Area", "Perimeter", "Neighbors Num", "Shape"]])

In [11]:
data[["Area", "Perimeter", "Neighbors Num", "Shape"]].corr().style.background_gradient(cmap='coolwarm')

,Area,Perimeter,Neighbors Num,Shape
Area,1.000000,0.946376,0.514222,0.188042
Perimeter,0.946376,1.000000,0.501700,0.472692
Neighbors Num,0.514222,0.501700,1.000000,0.119279
Shape,0.188042,0.472692,0.119279,1.000000


In [ ]:
image_path = r"C:\Users\victo\OneDrive - BGU\DATA\Hacat\6 Wells _5x_15min\_1\Pos0\*.tif"
mask_path = r"C:\Users\victo\OneDrive - BGU\DATA\Hacat\6 Wells _5x_15min\_1\Pos0\Mask2\*.png"

img_list = natsorted(glob.glob(image_path), key=lambda y: y.lower())
masks_list = natsorted(glob.glob(mask_path), key=lambda y: y.lower())
x, y, w, h = [0, 0, 800, 800]

df_list = []
for frame, mask_path in enumerate(masks_list[-20:]):
    # Calculate neighbors
    neighbors = calculate_neighbors(cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)[y:y+h, x:x+w])
    

    # Create DataFrame
    df_neighbors = create_dataframe(neighbors)
    df_neighbors["frame"] = frame
    df_list.append(df_neighbors)
    # break

features = pd.concat(df_list, ignore_index=True)

search_range = 7 #15
t = tp.link_df(features, search_range, memory=1)

Frame 19: 3044 trajectories present.


In [90]:
# search_range = 7 #15
# t = tp.link_df(features, search_range, memory=1)
# plt.imshow(cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)[y:y+h, x:x+w])
# tp.plot_traj(t)

In [101]:
# x, y, w, h = [0, 0, 800, 800]
# mask_image = cv2.imread(masks_list[-10], cv2.IMREAD_UNCHANGED)[y:y+h, x:x+w]
# img1 = cv2.imread(img_list[-10], cv2.IMREAD_UNCHANGED)[y:y+h, x:x+w]
# img2 = cv2.imread(img_list[-9], cv2.IMREAD_UNCHANGED)[y:y+h, x:x+w]
# flow = cv2.calcOpticalFlowFarneback(img1,img2, None, 0.5, 3, 
#         winsize=15, iterations=3, poly_n=5, poly_sigma=1.2, flags=0) 

# plt.figure()
# # clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
# # img_clahe = clahe.apply(img1)
# # plt.imshow(255-img_clahe, cmap='gray')
# overlay = cv2.addWeighted(cv2.cvtColor(img1, cv2.COLOR_GRAY2RGB), 0.5, plot.mask_rgb(mask_image), 0.4, 0)
# plt.imshow(overlay)

# step = 15
# xx = np.arange(0, flow.shape[1], step, dtype=np.int16)
# yy = np.arange(0, flow.shape[0], step, dtype=np.int16)
# plt.quiver(xx,yy, 
#     flow[::step, ::step, 0], -flow[::step, ::step, 1], 
#     color="w", scale=100, label="flow", alpha=.6)

# tp.plot_traj(t)

<Axes: xlabel='x [px]', ylabel='y [px]'>

In [92]:
t["dx"] = t.groupby("particle")["x"].diff()
t["dy"] = t.groupby("particle")["y"].diff()
t["displacement"] = np.sqrt(t["dx"]**2 + t["dy"]**2)
t.dropna(inplace=True)

In [96]:
import seaborn as sns
# data = t[t["Area"]>60].copy()
data = t.copy()
data["Shape"] = data["Perimeter"]**2/(4*np.pi*data["Area"])
sns.pairplot(data[["Area", "Perimeter", "Shape", "Neighbors Num", "displacement"]])

In [102]:
data[["Area", "Perimeter", "Shape", "Neighbors Num", "displacement"]].corr().style.background_gradient(cmap='coolwarm')

,Area,Perimeter,Shape,Neighbors Num,displacement
Area,1.000000,0.942801,0.207784,0.474773,-0.051676
Perimeter,0.942801,1.000000,0.500443,0.462450,-0.039247
Shape,0.207784,0.500443,1.000000,0.113627,0.027518
Neighbors Num,0.474773,0.462450,0.113627,1.000000,-0.008565
displacement,-0.051676,-0.039247,0.027518,-0.008565,1.000000


In [103]:
t

,Cell,x,y,Area,Perimeter,Neighbors Num,Neighbors,frame,particle,dx,dy,displacement
2861,151,425.164835,11.890110,91.0,35.935029,5,"[176, 156, 29, 30, 31]",1,31,0.931502,-3.809890,3.922111
2862,155,403.855072,14.753623,138.0,45.313708,6,"[193, 138, 235, 176, 28, 29]",1,2806,-2.500483,-4.157488,4.851507
2863,156,434.414966,16.156463,147.0,46.142136,7,"[33, 267, 176, 177, 151, 30, 31]",1,2810,-0.330376,-4.831115,4.842398
2864,162,116.605042,21.378151,238.0,60.183766,6,"[324, 9, 205, 175, 222, 127]",1,11,-1.287335,0.763801,1.496871
2865,163,238.688889,20.622222,315.0,71.698485,7,"[258, 197, 325, 141, 14, 15, 16]",1,2808,-2.476674,-3.602943,4.372084
...,...,...,...,...,...,...,...,...,...,...,...,...
58487,6590,333.285714,783.467532,154.0,45.556349,4,"[6538, 6687, 6429, 6607]",19,5151,-0.464286,-1.142762,1.233477
58488,6591,544.048780,782.304878,82.0,32.970563,6,"[6626, 6406, 6663, 6581, 6518, 6492]",19,5158,0.572590,-2.228455,2.300842
58489,6592,621.276471,783.935294,170.0,48.526912,5,"[6568, 6680, 6646, 6424, 6527]",19,4884,0.482212,0.365916,0.605329
58490,6625,514.528000,786.416000,125.0,43.798990,5,"[6662, 6728, 6539, 6674, 6526]",19,5161,0.126639,-1.774476,1.778989
